# Outlier diagnostics notebook (old vs new detector)

Aquest notebook serveix per:

- carregar el `*_pitch_preprocessed_debug.parquet`
- comparar **l'outlier detector antic** amb un **detector nou basat en context + forma**
- inspeccionar runs d'outliers i veure **per quin subcriteri** s'ha marcat cada tros
- visualitzar i escoltar una finestra temporal amb **dos subplots**:
  - dalt: outliers guardats / detector antic
  - baix: detector nou tunejat
- navegar per runs de:
  - `is_outlier_stored`
  - `is_outlier_old_recomputed`
  - `is_outlier_new`
  - submàscares del detector nou (`vel`, `shape`)
  - `too_long_to_interp`

**Detector antic**
- salt local per velocitat
- desviació respecte a mediana local

**Detector nou**
- salt local per velocitat (més permissiu)
- detecció de runs curts contextualment incoherents:
  - mida de l'excursió
  - semblança context esquerra/dreta
  - forma interna (`std`, `range`, canvis de signe, curvatura)
  - proximitat a gaps


In [ ]:

# --- paràmetres principals ---
RECORDING_ID = "srs_v1_bdn_sav"   # canvia-ho
TONIC_HZ = 146.83                 # canvia-ho
ROOT_DIR = "data/interim"

# si el detectem automàticament, no cal tocar-ho
AUDIO_PATH = None

# paràmetres "antics" del pipeline
OLD_MAX_VELOCITY_STS = 200.0
OLD_DEV_THRESH_CENTS = 600.0
OLD_EXPAND_NEIGHBORS = 1

# paràmetres "nous" inicials per comparar
NEW_MAX_VELOCITY_STS = 300.0
NEW_DEV_THRESH_CENTS = 600.0
NEW_EXPAND_NEIGHBORS = 1

In [ ]:

from pathlib import Path
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import soundfile as sf

from IPython.display import display, Audio, Javascript

try:
    from settings import PROJECT_ROOT
except Exception:
    PROJECT_ROOT = Path(".").resolve()

plt.rcParams["figure.figsize"] = (14, 5)

In [ ]:

def resolve_path(p):
    p = Path(p)
    if p.is_absolute():
        return p
    return Path(PROJECT_ROOT) / p

def get_debug_pitch_path(recording_id, root_dir="data/interim"):
    root_dir = resolve_path(root_dir)
    return root_dir / recording_id / "pitch" / f"{recording_id}_pitch_preprocessed_debug.parquet"

def find_audio_path(recording_id):
    corpus_root = resolve_path("data/corpus") / recording_id / "audio"
    if not corpus_root.exists():
        return None
    exts = [".wav", ".mp3", ".flac", ".ogg", ".m4a"]
    files = []
    for ext in exts:
        files.extend(sorted(corpus_root.glob(f"*{ext}")))
    return files[0] if files else None

debug_path = get_debug_pitch_path(RECORDING_ID, ROOT_DIR)
df_debug = pl.read_parquet(debug_path)

if AUDIO_PATH is None:
    AUDIO_PATH = find_audio_path(RECORDING_ID)

print("debug_path:", debug_path)
print("audio_path:", AUDIO_PATH)
print("shape:", df_debug.shape)
display(df_debug.head().to_pandas())

In [ ]:

def hz_to_cents(f_hz, tonic_hz):
    f_hz = np.asarray(f_hz, dtype=float)
    out = np.full_like(f_hz, np.nan, dtype=float)
    mask = np.isfinite(f_hz) & (f_hz > 0)
    out[mask] = 1200.0 * np.log2(f_hz[mask] / tonic_hz)
    return out

for col in ["f0_Hz", "f0_pchip", "f0_savgol_p3_w13", "f0_savgol_p3_w27", "f0_interpolated"]:
    if col in df_debug.columns and f"{col}_cents" not in df_debug.columns:
        df_debug = df_debug.with_columns(
            pl.Series(f"{col}_cents", hz_to_cents(df_debug[col].to_numpy(), TONIC_HZ))
        )

df_debug.columns

## 1. Funcions de diagnòstic dels outliers

La funció següent reprodueix la lògica de `remove_outliers(...)`, però retornant per separat:

- `is_outlier_vel_raw`: criteri de velocitat
- `is_outlier_dev_raw`: criteri de desviació respecte a mediana local
- `is_outlier_raw`: unió dels dos criteris abans d'expandir
- `is_outlier`: màscara final després d'expandir veïns
- `f0_clean`: columna d'entrada amb els outliers posats a `NaN`

In [ ]:
## 1. Funcions de diagnòstic

Aquí definim dues funcions de diagnòstic:

- `compute_outlier_diagnostics_old(...)`
  - reprodueix la lògica antiga del `remove_outliers(...)`
  - retorna per separat:
    - `is_outlier_vel_raw`
    - `is_outlier_dev_raw`
    - `is_outlier`
- `compute_outlier_diagnostics_new(...)`
  - implementa el detector nou:
    - `is_outlier_vel_raw`
    - `is_outlier_shape_raw`
    - `is_outlier`

Això permet comparar:
- outliers guardats al parquet
- detector antic recalculat
- detector nou tunejat


In [ ]:

from scipy.signal import medfilt

def find_true_runs(mask_bool):
    mask_bool = np.asarray(mask_bool, dtype=bool)
    runs = []
    in_run = False
    start = None

    for i, v in enumerate(mask_bool):
        if v and not in_run:
            start = i
            in_run = True
        elif not v and in_run:
            runs.append((start, i))  # end exclusiu
            in_run = False

    if in_run:
        runs.append((start, len(mask_bool)))
    return runs


def _fill_bridged_short_gaps(mask_bool: np.ndarray, max_gap: int) -> np.ndarray:
    if max_gap <= 0:
        return mask_bool

    mask = mask_bool.copy()
    runs = find_true_runs(mask)

    if len(runs) < 2:
        return mask

    for (s1, e1), (s2, e2) in zip(runs[:-1], runs[1:]):
        gap = s2 - e1
        if 0 < gap <= max_gap:
            mask[e1:s2] = True

    return mask


def _count_invalid_left(mask_valid: np.ndarray, idx_start: int) -> int:
    c = 0
    j = idx_start - 1
    while j >= 0 and not mask_valid[j]:
        c += 1
        j -= 1
    return c


def _count_invalid_right(mask_valid: np.ndarray, idx_end_exclusive: int) -> int:
    c = 0
    j = idx_end_exclusive
    while j < len(mask_valid) and not mask_valid[j]:
        c += 1
        j += 1
    return c


def _shape_stats(y: np.ndarray):
    n = len(y)

    if n == 0:
        return {
            "std": 0.0,
            "range": 0.0,
            "mean_d2": 0.0,
            "n_sign_changes": 0,
            "peak_pos_norm": 0.5,
            "trough_pos_norm": 0.5,
        }

    y = np.asarray(y, dtype=float)

    run_std = float(np.std(y)) if n > 1 else 0.0
    run_range = float(np.max(y) - np.min(y)) if n > 1 else 0.0

    if n < 3:
        return {
            "std": run_std,
            "range": run_range,
            "mean_d2": 0.0,
            "n_sign_changes": 0,
            "peak_pos_norm": 0.5,
            "trough_pos_norm": 0.5,
        }

    d1 = np.diff(y)
    d1_nz = d1[np.abs(d1) > 1e-9]

    if len(d1_nz) <= 1:
        n_sign_changes = 0
    else:
        signs = np.sign(d1_nz)
        n_sign_changes = int(np.sum(signs[1:] != signs[:-1]))

    d2 = np.diff(d1)
    mean_d2 = float(np.mean(d2)) if len(d2) > 0 else 0.0

    peak_pos = int(np.argmax(y))
    trough_pos = int(np.argmin(y))
    denom = max(1, n - 1)

    return {
        "std": run_std,
        "range": run_range,
        "mean_d2": mean_d2,
        "n_sign_changes": n_sign_changes,
        "peak_pos_norm": peak_pos / denom,
        "trough_pos_norm": trough_pos / denom,
    }


def _detect_context_incoherent_runs(
    f_cents_full: np.ndarray,
    mask_valid_full: np.ndarray,
    is_outlier_seed_full: np.ndarray,
    max_run_len: int = 12,
    context_len: int = 4,
    excursion_thresh_cents: float = 180.0,
    context_match_thresh_cents: float = 140.0,
    max_internal_std_cents: float = 120.0,
    max_internal_range_cents: float = 260.0,
    max_slope_sign_changes: int = 1,
    max_edge_jump_cents_per_sample: float = 140.0,
    gap_bonus_invalid: int = 1,
):
    N = len(f_cents_full)
    out = np.zeros(N, dtype=bool)

    valid_runs = find_true_runs(mask_valid_full)

    for s, e in valid_runs:
        if e <= s:
            continue

        for rs in range(s, e):
            for re in range(rs + 1, min(e, rs + max_run_len) + 1):
                y = f_cents_full[rs:re]
                if len(y) == 0 or not np.all(np.isfinite(y)):
                    continue

                left_ctx = f_cents_full[max(s, rs - context_len):rs]
                right_ctx = f_cents_full[re:min(e, re + context_len)]

                left_ctx = left_ctx[np.isfinite(left_ctx)]
                right_ctx = right_ctx[np.isfinite(right_ctx)]

                if len(left_ctx) < 2 or len(right_ctx) < 2:
                    continue

                med_left = float(np.median(left_ctx))
                med_mid = float(np.median(y))
                med_right = float(np.median(right_ctx))

                excursion_left = abs(med_mid - med_left)
                excursion_right = abs(med_mid - med_right)
                context_match = abs(med_left - med_right)

                if not (
                    excursion_left >= excursion_thresh_cents
                    and excursion_right >= excursion_thresh_cents
                    and context_match <= context_match_thresh_cents
                ):
                    continue

                stats = _shape_stats(y)

                context_center = 0.5 * (med_left + med_right)
                is_up_island = med_mid > context_center
                is_down_island = med_mid < context_center

                jump_in = abs(y[0] - med_left)
                jump_out = abs(y[-1] - med_right)

                n_invalid_left = _count_invalid_left(mask_valid_full, rs)
                n_invalid_right = _count_invalid_right(mask_valid_full, re)

                jump_in_per_sample = jump_in / max(1, 1 + n_invalid_left)
                jump_out_per_sample = jump_out / max(1, 1 + n_invalid_right)

                touches_gap = (n_invalid_left >= gap_bonus_invalid) or (n_invalid_right >= gap_bonus_invalid)

                seed_nearby = False
                if rs > 0 and is_outlier_seed_full[rs - 1]:
                    seed_nearby = True
                if re < N and is_outlier_seed_full[re]:
                    seed_nearby = True
                if np.any(is_outlier_seed_full[rs:re]):
                    seed_nearby = True

                shape_bad = False

                if (
                    stats["std"] <= max_internal_std_cents
                    and stats["range"] <= max_internal_range_cents
                ):
                    shape_bad = True

                if stats["n_sign_changes"] > max_slope_sign_changes:
                    shape_bad = True

                if is_up_island and stats["mean_d2"] > 0:
                    shape_bad = True

                if is_down_island and stats["mean_d2"] < 0:
                    shape_bad = True

                if is_up_island and (stats["peak_pos_norm"] < 0.15 or stats["peak_pos_norm"] > 0.85):
                    shape_bad = True
                if is_down_island and (stats["trough_pos_norm"] < 0.15 or stats["trough_pos_norm"] > 0.85):
                    shape_bad = True

                edge_bad = (
                    jump_in_per_sample > max_edge_jump_cents_per_sample
                    or jump_out_per_sample > max_edge_jump_cents_per_sample
                )

                gap_bad = False
                if touches_gap:
                    if (
                        stats["std"] <= (max_internal_std_cents * 1.35)
                        and context_match <= (context_match_thresh_cents * 0.9)
                    ):
                        gap_bad = True

                if shape_bad or edge_bad or gap_bad or seed_nearby:
                    out[rs:re] = True

    return out


def compute_outlier_diagnostics_old(
    df: pl.DataFrame,
    col_time: str = "time_rel_sec",
    col_f0: str = "f0_interpolated",
    col_too_long: str = "too_long_to_interp",
    max_velocity_sts: float = 200.0,
    dev_thresh_cents: float = 600.0,
    expand_neighbors: int = 1,
    f_ref=None,
):
    t = df[col_time].to_numpy()
    f = df[col_f0].to_numpy().astype(float)
    too_long = df[col_too_long].fill_null(False).to_numpy().astype(bool)

    N = len(f)
    mask_valid = (f > 0) & (~too_long)
    valid_idx = np.where(mask_valid)[0]

    if len(valid_idx) < 3:
        return {
            "is_outlier_vel_raw": np.zeros(N, dtype=bool),
            "is_outlier_dev_raw": np.zeros(N, dtype=bool),
            "is_outlier": np.zeros(N, dtype=bool),
            "vel_jump_cents": np.full(N, np.nan),
            "vel_max_allowed_cents": np.full(N, np.nan),
            "median_local": np.full(N, np.nan),
            "dev_cents": np.full(N, np.nan),
            "f0_clean": f.copy(),
        }

    if f_ref is None:
        f_ref = np.nanmedian(f[mask_valid])

    f_cents = np.full(N, np.nan, dtype=float)
    f_cents[mask_valid] = 1200.0 * np.log2(f[mask_valid] / f_ref)

    max_vel_cents = max_velocity_sts * 100.0

    is_outlier_vel = np.zeros(N, dtype=bool)
    vel_jump_cents = np.full(N, np.nan, dtype=float)
    vel_max_allowed_cents = np.full(N, np.nan, dtype=float)

    prev_idx = valid_idx[0]
    prev_cents = f_cents[prev_idx]
    prev_time = t[prev_idx]

    for idx in valid_idx[1:]:
        curr_cents = f_cents[idx]
        curr_time = t[idx]

        dt = curr_time - prev_time
        if dt <= 0:
            prev_idx = idx
            prev_cents = curr_cents
            prev_time = curr_time
            continue

        dcents = abs(curr_cents - prev_cents)
        max_d_allowed = max_vel_cents * dt

        vel_jump_cents[idx] = dcents
        vel_max_allowed_cents[idx] = max_d_allowed

        if dcents > max_d_allowed:
            is_outlier_vel[idx] = True

        prev_idx = idx
        prev_cents = curr_cents
        prev_time = curr_time

    f_cents_valid = f_cents[valid_idx]

    kernel_size = 31
    if kernel_size > len(f_cents_valid):
        kernel_size = len(f_cents_valid) if len(f_cents_valid) % 2 == 1 else len(f_cents_valid) - 1
    if kernel_size < 3:
        kernel_size = 3

    median_local_valid = medfilt(f_cents_valid, kernel_size=kernel_size)
    dev_valid = np.abs(f_cents_valid - median_local_valid)
    is_outlier_dev_local = dev_valid > dev_thresh_cents

    median_local = np.full(N, np.nan, dtype=float)
    dev_cents = np.full(N, np.nan, dtype=float)
    is_outlier_dev = np.zeros(N, dtype=bool)

    median_local[valid_idx] = median_local_valid
    dev_cents[valid_idx] = dev_valid
    is_outlier_dev[valid_idx] = is_outlier_dev_local

    is_outlier = is_outlier_vel | is_outlier_dev

    if expand_neighbors > 0:
        is_outlier_expanded = is_outlier.copy()
        for shift in range(1, expand_neighbors + 1):
            is_outlier_expanded[:-shift] |= is_outlier[shift:]
            is_outlier_expanded[shift:] |= is_outlier[:-shift]
        is_outlier = is_outlier_expanded

    is_outlier &= mask_valid

    f_clean = f.copy()
    f_clean[is_outlier] = np.nan

    return {
        "is_outlier_vel_raw": is_outlier_vel,
        "is_outlier_dev_raw": is_outlier_dev,
        "is_outlier": is_outlier,
        "vel_jump_cents": vel_jump_cents,
        "vel_max_allowed_cents": vel_max_allowed_cents,
        "median_local": median_local,
        "dev_cents": dev_cents,
        "f0_clean": f_clean,
    }


def compute_outlier_diagnostics_new(
    df: pl.DataFrame,
    col_time: str = "time_rel_sec",
    col_f0: str = "f0_interpolated",
    col_too_long: str = "too_long_to_interp",
    max_velocity_sts: float = 260.0,
    dev_thresh_cents: float = 600.0,
    expand_neighbors: int = 1,
    f_ref=None,
):
    t = df[col_time].to_numpy()
    f = df[col_f0].to_numpy().astype(float)
    too_long = df[col_too_long].fill_null(False).to_numpy().astype(bool)

    N = len(f)
    mask_valid = (f > 0) & (~too_long)
    valid_idx = np.where(mask_valid)[0]

    if len(valid_idx) < 3:
        return {
            "is_outlier_vel_raw": np.zeros(N, dtype=bool),
            "is_outlier_shape_raw": np.zeros(N, dtype=bool),
            "is_outlier": np.zeros(N, dtype=bool),
            "vel_jump_cents": np.full(N, np.nan),
            "vel_max_allowed_cents": np.full(N, np.nan),
            "f0_clean": f.copy(),
        }

    if f_ref is None:
        f_ref = np.nanmedian(f[mask_valid])

    f_cents = np.full(N, np.nan, dtype=float)
    f_cents[mask_valid] = 1200.0 * np.log2(f[mask_valid] / f_ref)

    max_vel_cents = max_velocity_sts * 100.0

    dt_all = np.diff(t[np.isfinite(t)])
    dt_median = np.median(dt_all) if len(dt_all) > 0 else 0.01
    if not np.isfinite(dt_median) or dt_median <= 0:
        dt_median = 0.01

    max_jump_cents_per_sample = max_vel_cents * dt_median

    is_outlier_vel = np.zeros(N, dtype=bool)
    vel_jump_cents = np.full(N, np.nan, dtype=float)
    vel_max_allowed_cents = np.full(N, np.nan, dtype=float)

    prev_idx = valid_idx[0]
    prev_cents = f_cents[prev_idx]
    prev_time = t[prev_idx]

    for idx in valid_idx[1:]:
        curr_cents = f_cents[idx]
        curr_time = t[idx]

        dt = curr_time - prev_time
        sample_gap = idx - prev_idx

        if dt <= 0 or sample_gap <= 0:
            prev_idx = idx
            prev_cents = curr_cents
            prev_time = curr_time
            continue

        dcents = abs(curr_cents - prev_cents)

        max_d_allowed_time = max_vel_cents * dt
        max_d_allowed_samples = max_jump_cents_per_sample * sample_gap
        max_d_allowed = max(max_d_allowed_time, max_d_allowed_samples)

        vel_jump_cents[idx] = dcents
        vel_max_allowed_cents[idx] = max_d_allowed

        if dcents > max_d_allowed:
            is_outlier_vel[idx] = True

        prev_idx = idx
        prev_cents = curr_cents
        prev_time = curr_time

    is_outlier_vel = _fill_bridged_short_gaps(is_outlier_vel, max_gap=6)

    is_outlier_shape = _detect_context_incoherent_runs(
        f_cents_full=f_cents,
        mask_valid_full=mask_valid,
        is_outlier_seed_full=is_outlier_vel,
        max_run_len=12,
        context_len=4,
        excursion_thresh_cents=max(160.0, 0.45 * dev_thresh_cents),
        context_match_thresh_cents=140.0,
        max_internal_std_cents=120.0,
        max_internal_range_cents=260.0,
        max_slope_sign_changes=1,
        max_edge_jump_cents_per_sample=140.0,
        gap_bonus_invalid=1,
    )

    is_outlier = is_outlier_vel | is_outlier_shape

    if expand_neighbors > 0:
        is_outlier_expanded = is_outlier.copy()
        for shift in range(1, expand_neighbors + 1):
            is_outlier_expanded[:-shift] |= is_outlier[shift:]
            is_outlier_expanded[shift:] |= is_outlier[:-shift]
        is_outlier = is_outlier_expanded

    is_outlier &= mask_valid

    f_clean = f.copy()
    f_clean[is_outlier] = np.nan

    return {
        "is_outlier_vel_raw": is_outlier_vel,
        "is_outlier_shape_raw": is_outlier_shape,
        "is_outlier": is_outlier,
        "vel_jump_cents": vel_jump_cents,
        "vel_max_allowed_cents": vel_max_allowed_cents,
        "f0_clean": f_clean,
    }



diag_old = compute_outlier_diagnostics_old(
    df_debug,
    col_f0="f0_interpolated",
    max_velocity_sts=OLD_MAX_VELOCITY_STS,
    dev_thresh_cents=OLD_DEV_THRESH_CENTS,
    expand_neighbors=OLD_EXPAND_NEIGHBORS,
    f_ref=TONIC_HZ,
)

diag_new = compute_outlier_diagnostics_new(
    df_debug,
    col_f0="f0_interpolated",
    max_velocity_sts=NEW_MAX_VELOCITY_STS,
    dev_thresh_cents=NEW_DEV_THRESH_CENTS,
    expand_neighbors=NEW_EXPAND_NEIGHBORS,
    f_ref=TONIC_HZ,
)

df_diag = df_debug.with_columns([
    pl.Series("is_outlier_stored", df_debug["is_outlier"].fill_null(False).to_numpy().astype(bool)),
    pl.Series("is_outlier_old_recomputed", diag_old["is_outlier"]),
    pl.Series("is_outlier_old_vel_raw", diag_old["is_outlier_vel_raw"]),
    pl.Series("is_outlier_old_dev_raw", diag_old["is_outlier_dev_raw"]),
    pl.Series("old_vel_jump_cents", diag_old["vel_jump_cents"]),
    pl.Series("old_vel_max_allowed_cents", diag_old["vel_max_allowed_cents"]),
    pl.Series("old_median_local_cents", diag_old["median_local"]),
    pl.Series("old_dev_cents", diag_old["dev_cents"]),
    pl.Series("is_outlier_new", diag_new["is_outlier"]),
    pl.Series("is_outlier_new_vel_raw", diag_new["is_outlier_vel_raw"]),
    pl.Series("is_outlier_new_shape_raw", diag_new["is_outlier_shape_raw"]),
    pl.Series("new_vel_jump_cents", diag_new["vel_jump_cents"]),
    pl.Series("new_vel_max_allowed_cents", diag_new["vel_max_allowed_cents"]),
])

summary = {
    "stored": int(df_diag["is_outlier_stored"].sum()),
    "old_recomputed": int(df_diag["is_outlier_old_recomputed"].sum()),
    "old_vel_raw": int(df_diag["is_outlier_old_vel_raw"].sum()),
    "old_dev_raw": int(df_diag["is_outlier_old_dev_raw"].sum()),
    "new_total": int(df_diag["is_outlier_new"].sum()),
    "new_vel_raw": int(df_diag["is_outlier_new_vel_raw"].sum()),
    "new_shape_raw": int(df_diag["is_outlier_new_shape_raw"].sum()),
}
summary


In [ ]:

def find_true_runs(mask_bool):
    mask_bool = np.asarray(mask_bool, dtype=bool)
    runs = []
    in_run = False
    start = None

    for i, v in enumerate(mask_bool):
        if v and not in_run:
            start = i
            in_run = True
        elif not v and in_run:
            runs.append((start, i))
            in_run = False

    if in_run:
        runs.append((start, len(mask_bool)))
    return runs

def build_run_table(df: pl.DataFrame, mask_col: str, prefix: str = "old"):
    t = df["time_rel_sec"].to_numpy()
    runs = find_true_runs(df[mask_col].fill_null(False).to_numpy().astype(bool))

    rows = []
    for k, (s, e) in enumerate(runs):
        rows.append({
            "run_id": k,
            "start_idx": s,
            "end_idx_exclusive": e,
            "n_samples": e - s,
            "t_start": float(t[s]),
            "t_end": float(t[e - 1]),
            "t_mid": float(0.5 * (t[s] + t[e - 1])),
            "any_vel_raw": bool(df[f"is_outlier_{prefix}_vel_raw"][s:e].any()) if f"is_outlier_{prefix}_vel_raw" in df.columns else None,
            "any_dev_raw": bool(df[f"is_outlier_{prefix}_dev_raw"][s:e].any()) if f"is_outlier_{prefix}_dev_raw" in df.columns else None,
            "max_vel_jump_cents": float(np.nanmax(df[f"{prefix}_vel_jump_cents"][s:e].to_numpy())) if f"{prefix}_vel_jump_cents" in df.columns else np.nan,
            "max_vel_allowed_cents": float(np.nanmax(df[f"{prefix}_vel_max_allowed_cents"][s:e].to_numpy())) if f"{prefix}_vel_max_allowed_cents" in df.columns else np.nan,
            "max_dev_cents": float(np.nanmax(df[f"{prefix}_dev_cents"][s:e].to_numpy())) if f"{prefix}_dev_cents" in df.columns else np.nan,
        })
    return pd.DataFrame(rows)

runs_stored = build_run_table(df_diag, "is_outlier_stored", prefix="old")
runs_old = build_run_table(df_diag, "is_outlier_old_recomputed", prefix="old")
runs_new = build_run_table(df_diag, "is_outlier_new", prefix="new")

print("stored runs:", len(runs_stored))
display(runs_stored.head(20))

In [ ]:

def build_run_table(df: pl.DataFrame, mask_col: str, prefix: str = "old"):
    t = df["time_rel_sec"].to_numpy()
    runs = find_true_runs(df[mask_col].fill_null(False).to_numpy().astype(bool))

    rows = []
    for k, (s, e) in enumerate(runs):
        row = {
            "run_id": k,
            "start_idx": s,
            "end_idx_exclusive": e,
            "n_samples": e - s,
            "t_start": float(t[s]),
            "t_end": float(t[e - 1]),
            "t_mid": float(0.5 * (t[s] + t[e - 1])),
            "max_vel_jump_cents": float(np.nanmax(df[f"{prefix}_vel_jump_cents"][s:e].to_numpy())) if f"{prefix}_vel_jump_cents" in df.columns else np.nan,
            "max_vel_allowed_cents": float(np.nanmax(df[f"{prefix}_vel_max_allowed_cents"][s:e].to_numpy())) if f"{prefix}_vel_max_allowed_cents" in df.columns else np.nan,
        }

        if f"is_outlier_{prefix}_vel_raw" in df.columns:
            row["any_vel_raw"] = bool(df[f"is_outlier_{prefix}_vel_raw"][s:e].any())
        else:
            row["any_vel_raw"] = None

        if f"is_outlier_{prefix}_dev_raw" in df.columns:
            row["any_dev_raw"] = bool(df[f"is_outlier_{prefix}_dev_raw"][s:e].any())
        else:
            row["any_dev_raw"] = None

        if f"is_outlier_{prefix}_shape_raw" in df.columns:
            row["any_shape_raw"] = bool(df[f"is_outlier_{prefix}_shape_raw"][s:e].any())
        else:
            row["any_shape_raw"] = None

        if f"{prefix}_dev_cents" in df.columns:
            row["max_dev_cents"] = float(np.nanmax(df[f"{prefix}_dev_cents"][s:e].to_numpy()))
        else:
            row["max_dev_cents"] = np.nan

        rows.append(row)

    return pd.DataFrame(rows)

runs_stored = build_run_table(df_diag, "is_outlier_stored", prefix="old")
runs_old = build_run_table(df_diag, "is_outlier_old_recomputed", prefix="old")
runs_new = build_run_table(df_diag, "is_outlier_new", prefix="new")

print("stored runs:", len(runs_stored))
display(runs_stored.head(20))


## 3. Viewer comparatiu: outliers antics vs tuned

A dalt es pinta `is_outlier_stored`.

A baix es pinta `is_outlier_new`, calculat amb els controls de paràmetres.

També es mostren:
- `is_outlier_old_vel_raw` o `is_outlier_new_vel_raw`
- `is_outlier_old_dev_raw` o `is_outlier_new_dev_raw`
si els actives.

El reproductor reprodueix exactament la finestra visible.

In [ ]:
## 3. Viewer comparatiu: detector antic vs detector nou

A dalt es pinta `is_outlier_stored` i, opcionalment, les submàscares del detector antic:
- `is_outlier_old_vel_raw`
- `is_outlier_old_dev_raw`

A baix es pinta `is_outlier_new` i, opcionalment, les submàscares del detector nou:
- `is_outlier_new_vel_raw`
- `is_outlier_new_shape_raw`

El reproductor reprodueix exactament la finestra visible.


In [ ]:

def extract_audio_window(audio_path, t_start, window_sec):
    if audio_path is None:
        return None, None, t_start, t_start + window_sec, None

    audio_path = Path(audio_path)
    y, sr = sf.read(audio_path)
    audio_dur = len(y) / sr

    t0 = max(0.0, float(t_start))
    t1 = min(float(t_start + window_sec), audio_dur)

    s0 = int(round(t0 * sr))
    s1 = int(round(t1 * sr))
    y_win = y[s0:s1]
    return y_win, sr, t0, t1, audio_dur


def get_runs_from_mask(df: pl.DataFrame, mask_col: str, time_col: str = "time_rel_sec"):
    if mask_col not in df.columns:
        return []
    t = df[time_col].to_numpy()
    mask = df[mask_col].fill_null(False).to_numpy().astype(bool)
    runs = find_true_runs(mask)

    out = []
    for s, e in runs:
        x0 = t[s]
        x1 = t[e - 1]
        left = 0.5 * (t[s - 1] + t[s]) if s > 0 else x0
        right = 0.5 * (t[e - 1] + t[e]) if e < len(t) else x1
        x_mid = float(0.5 * (left + right))
        out.append({
            "start_idx": s,
            "end_idx": e,
            "x_left": float(left),
            "x_right": float(right),
            "x_mid": x_mid,
            "n_samples": int(e - s),
        })
    return out


def make_diag_df(df_base, max_velocity_sts, dev_thresh_cents, expand_neighbors):
    diag_new = compute_outlier_diagnostics_new(
        df_base,
        col_f0="f0_interpolated",
        max_velocity_sts=max_velocity_sts,
        dev_thresh_cents=dev_thresh_cents,
        expand_neighbors=expand_neighbors,
        f_ref=TONIC_HZ,
    )
    diag_old_local = compute_outlier_diagnostics_old(
        df_base,
        col_f0="f0_interpolated",
        max_velocity_sts=OLD_MAX_VELOCITY_STS,
        dev_thresh_cents=OLD_DEV_THRESH_CENTS,
        expand_neighbors=OLD_EXPAND_NEIGHBORS,
        f_ref=TONIC_HZ,
    )

    return df_base.with_columns([
        pl.Series("is_outlier_stored", df_base["is_outlier"].fill_null(False).to_numpy().astype(bool)),
        pl.Series("is_outlier_old_recomputed", diag_old_local["is_outlier"]),
        pl.Series("is_outlier_old_vel_raw", diag_old_local["is_outlier_vel_raw"]),
        pl.Series("is_outlier_old_dev_raw", diag_old_local["is_outlier_dev_raw"]),
        pl.Series("is_outlier_new", diag_new["is_outlier"]),
        pl.Series("is_outlier_new_vel_raw", diag_new["is_outlier_vel_raw"]),
        pl.Series("is_outlier_new_shape_raw", diag_new["is_outlier_shape_raw"]),
    ])


In [ ]:

def plot_compare_outliers(
    df: pl.DataFrame,
    t_start: float,
    t_end: float,
    show_counts: bool = True,
    show_old_vel: bool = True,
    show_old_dev: bool = True,
    show_new_vel: bool = True,
    show_new_shape: bool = True,
):
    df_win = (
        df.filter((pl.col("time_rel_sec") >= t_start) & (pl.col("time_rel_sec") <= t_end))
          .sort("time_rel_sec")
    )
    if df_win.height == 0:
        print("No hi ha dades en aquest rang.")
        return

    t = df_win["time_rel_sec"].to_numpy()
    hz = df_win["f0_Hz_cents"].to_numpy() if "f0_Hz_cents" in df_win.columns else None
    pchip = df_win["f0_pchip_cents"].to_numpy() if "f0_pchip_cents" in df_win.columns else None
    sg13 = df_win["f0_savgol_p3_w13_cents"].to_numpy() if "f0_savgol_p3_w13_cents" in df_win.columns else None
    sg27 = df_win["f0_savgol_p3_w27_cents"].to_numpy() if "f0_savgol_p3_w27_cents" in df_win.columns else None
    interp = df_win["f0_interpolated_cents"].to_numpy() if "f0_interpolated_cents" in df_win.columns else None

    fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)

    def draw_base(ax):
        if sg27 is not None:
            ax.plot(t, sg27, color="#9467bd", linewidth=3.5, alpha=0.75, zorder=5, label="sg27")
        if sg13 is not None:
            ax.plot(t, sg13, color="#2ca02c", linewidth=3.0, alpha=0.85, zorder=10, label="sg13")
        if pchip is not None:
            ax.plot(t, pchip, color="#d62728", linewidth=1.2, alpha=1, zorder=30, label="pchip")
        if interp is not None:
            ax.plot(t, interp, color="#8c564b", linewidth=1.0, alpha=0.45, zorder=32, label="interp")
        if hz is not None:
            ax.plot(t, hz, color="#001aff", linewidth=1.2, alpha=1, zorder=40, label="f0_Hz")

    def draw_mask(ax, mask_col, color, label, alpha=0.18, linestyle="--", y_text=None):
        if mask_col not in df_win.columns:
            return
        mask = df_win[mask_col].fill_null(False).to_numpy().astype(bool)
        runs = find_true_runs(mask)

        line_labeled = False
        region_labeled = False

        for s, e in runs:
            run_len = e - s
            if run_len == 1:
                x_mid = t[s]
                ax.axvline(
                    x=t[s],
                    color=color,
                    linestyle=linestyle,
                    linewidth=1.2,
                    alpha=alpha,
                    label=f"{label} (punt)" if not line_labeled else None,
                    zorder=20,
                )
                line_labeled = True
            else:
                x0 = t[s]
                x1 = t[e - 1]
                left = 0.5 * (t[s - 1] + t[s]) if s > 0 else x0
                right = 0.5 * (t[e - 1] + t[e]) if e < len(t) else x1
                x_mid = 0.5 * (left + right)
                ax.axvspan(
                    left, right,
                    color=color,
                    alpha=alpha,
                    label=f"{label} (regió)" if not region_labeled else None,
                    zorder=15,
                )
                region_labeled = True

            if show_counts and y_text is not None:
                ax.text(
                    x_mid, y_text, str(run_len),
                    ha="center", va="center", fontsize=9, color=color,
                    bbox=dict(boxstyle="round,pad=0.18", facecolor="white", edgecolor=color, alpha=0.85),
                    zorder=50, clip_on=False,
                )

    for ax in axes:
        draw_base(ax)

    for ax, main_mask, aux1_mask, aux2_mask, aux1_flag, aux2_flag, title, aux2_label in [
        (axes[0], "is_outlier_stored", "is_outlier_old_vel_raw", "is_outlier_old_dev_raw", show_old_vel, show_old_dev, "Outliers guardats / detector antic", "dev"),
        (axes[1], "is_outlier_new", "is_outlier_new_vel_raw", "is_outlier_new_shape_raw", show_new_vel, show_new_shape, "Detector nou tunejat", "shape"),
    ]:
        ymin, ymax = ax.get_ylim()
        yrange = ymax - ymin if np.isfinite(ymax - ymin) and ymax > ymin else 1.0
        y_top = ymax - 0.06 * yrange
        y_mid = ymax - 0.12 * yrange
        y_low = ymax - 0.18 * yrange

        draw_mask(ax, main_mask, "#0055ff", "outlier", alpha=0.18, y_text=y_mid)
        if aux1_flag:
            draw_mask(ax, aux1_mask, "#ff7f0e", "vel", alpha=0.13, y_text=y_top)
        if aux2_flag:
            draw_mask(ax, aux2_mask, "#2ca02c", aux2_label, alpha=0.10, y_text=y_low)

        ax.set_ylabel("cents")
        ax.set_title(title)
        ax.grid(True, alpha=0.25)
        ax.legend(loc="best")

    axes[-1].set_xlabel("time_rel_sec")
    plt.tight_layout()
    plt.show()


In [ ]:

def interactive_outlier_comparison_viewer(df_base: pl.DataFrame, audio_path, tonic_hz: float):
    t_min = float(df_base["time_rel_sec"].min())
    t_max = float(df_base["time_rel_sec"].max())

    if audio_path is not None:
        info = sf.info(str(audio_path))
        audio_dur = float(info.frames) / float(info.samplerate)
        global_max = min(t_max, audio_dur)
    else:
        audio_dur = None
        global_max = t_max

    global_min = max(0.0, t_min)

    window_slider = widgets.FloatSlider(
        value=5.0, min=0.5, max=max(0.5, min(30.0, global_max - global_min)),
        step=0.25, description="finestra (s)", continuous_update=False,
        readout_format=".2f", style={"description_width": "initial"},
        layout=widgets.Layout(width="650px"),
    )
    start_slider = widgets.FloatSlider(
        value=global_min, min=global_min, max=max(global_min, global_max - 5.0),
        step=0.25, description="inici (s)", continuous_update=False,
        readout_format=".2f", style={"description_width": "initial"},
        layout=widgets.Layout(width="650px"),
    )

    prev_button = widgets.Button(description="← finestra anterior", layout=widgets.Layout(width="180px"))
    next_button = widgets.Button(description="següent finestra →", layout=widgets.Layout(width="180px"))

    max_vel_slider = widgets.FloatSlider(
        value=NEW_MAX_VELOCITY_STS, min=20.0, max=600.0, step=5.0,
        description="new max_velocity_sts", continuous_update=False,
        style={"description_width": "initial"}, layout=widgets.Layout(width="420px"),
    )
    dev_slider = widgets.FloatSlider(
        value=NEW_DEV_THRESH_CENTS, min=50.0, max=1200.0, step=10.0,
        description="new dev_thresh_cents", continuous_update=False,
        style={"description_width": "initial"}, layout=widgets.Layout(width="420px"),
    )
    expand_slider = widgets.IntSlider(
        value=NEW_EXPAND_NEIGHBORS, min=0, max=6, step=1,
        description="new expand_neighbors", continuous_update=False,
        style={"description_width": "initial"}, layout=widgets.Layout(width="320px"),
    )

    nav_dropdown = widgets.Dropdown(
        options=[
            ("stored outlier", "is_outlier_stored"),
            ("old recomputed", "is_outlier_old_recomputed"),
            ("new outlier", "is_outlier_new"),
            ("old vel raw", "is_outlier_old_vel_raw"),
            ("old dev raw", "is_outlier_old_dev_raw"),
            ("new vel raw", "is_outlier_new_vel_raw"),
            ("new shape raw", "is_outlier_new_shape_raw"),
            ("too_long_to_interp", "too_long_to_interp"),
        ],
        value="is_outlier_stored",
        description="navegar",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="280px"),
    )
    prev_run_button = widgets.Button(description="← run prev", layout=widgets.Layout(width="160px"))
    next_run_button = widgets.Button(description="run next →", layout=widgets.Layout(width="160px"))

    autoplay_checkbox = widgets.Checkbox(value=False, description="autoplay àudio", indent=False)
    overlay_counts_checkbox = widgets.Checkbox(value=True, description="mostrar n samples", indent=False)
    old_vel_checkbox = widgets.Checkbox(value=True, description="mostrar old vel", indent=False)
    old_dev_checkbox = widgets.Checkbox(value=True, description="mostrar old dev", indent=False)
    new_vel_checkbox = widgets.Checkbox(value=True, description="mostrar new vel", indent=False)
    new_shape_checkbox = widgets.Checkbox(value=True, description="mostrar new shape", indent=False)

    info_box = widgets.Output()
    plot_box = widgets.Output()
    audio_box = widgets.Output()

    state = {"df_view": None, "runs_map": {}}

    def _sync_start_bounds(*args):
        w = float(window_slider.value)
        start_slider.max = max(global_min, global_max - w)
        if start_slider.value > start_slider.max:
            start_slider.value = start_slider.max

    def _recompute_df():
        df_view = make_diag_df(
            df_base,
            max_velocity_sts=float(max_vel_slider.value),
            dev_thresh_cents=float(dev_slider.value),
            expand_neighbors=int(expand_slider.value),
        )
        state["df_view"] = df_view
        state["runs_map"] = {
            col: get_runs_from_mask(df_view, col)
            for _, col in nav_dropdown.options
            if col in df_view.columns
        }
        return df_view

    def _center_window_on_x(x_mid):
        w = float(window_slider.value)
        new_start = x_mid - 0.5 * w
        new_start = max(global_min, min(new_start, start_slider.max))
        start_slider.value = new_start

    def _jump_to_run(direction=1):
        col = nav_dropdown.value
        runs = state["runs_map"].get(col, [])
        if not runs:
            return
        current_t0 = float(start_slider.value)
        current_t1 = min(current_t0 + float(window_slider.value), global_max)
        current_center = 0.5 * (current_t0 + current_t1)
        mids = np.array([r["x_mid"] for r in runs], dtype=float)

        if direction > 0:
            idxs = np.where(mids > current_center + 1e-12)[0]
            idx = int(idxs[0]) if len(idxs) > 0 else len(runs) - 1
        else:
            idxs = np.where(mids < current_center - 1e-12)[0]
            idx = int(idxs[-1]) if len(idxs) > 0 else 0
        _center_window_on_x(runs[idx]["x_mid"])

    def _go_prev(_):
        w = float(window_slider.value)
        start_slider.value = max(global_min, float(start_slider.value) - w)

    def _go_next(_):
        w = float(window_slider.value)
        start_slider.value = min(start_slider.max, float(start_slider.value) + w)

    prev_button.on_click(_go_prev)
    next_button.on_click(_go_next)
    prev_run_button.on_click(lambda _: _jump_to_run(-1))
    next_run_button.on_click(lambda _: _jump_to_run(1))
    window_slider.observe(_sync_start_bounds, names="value")

    def _refresh(*args):
        df_view = _recompute_df()

        t0 = float(start_slider.value)
        w = float(window_slider.value)
        t1 = min(t0 + w, global_max)

        info_box.clear_output(wait=True)
        plot_box.clear_output(wait=True)
        audio_box.clear_output(wait=True)

        with info_box:
            print(f"Finestra visible: {t0:.2f}s → {t1:.2f}s")
            print(f"old params: velocity={OLD_MAX_VELOCITY_STS}, dev={OLD_DEV_THRESH_CENTS}, expand={OLD_EXPAND_NEIGHBORS}")
            print(f"new params: velocity={max_vel_slider.value}, dev={dev_slider.value}, expand={expand_slider.value}")
            print(f"stored outliers: {int(df_view['is_outlier_stored'].sum())}")
            print(f"old recomputed: {int(df_view['is_outlier_old_recomputed'].sum())}")
            print(f"new outliers: {int(df_view['is_outlier_new'].sum())}")
            nav_col = nav_dropdown.value
            print(f"runs per '{nav_col}': {len(state['runs_map'].get(nav_col, []))}")

        with plot_box:
            plot_compare_outliers(
                df_view,
                t_start=t0,
                t_end=t1,
                show_counts=overlay_counts_checkbox.value,
                show_old_vel=old_vel_checkbox.value,
                show_old_dev=old_dev_checkbox.value,
                show_new_vel=new_vel_checkbox.value,
                show_new_shape=new_shape_checkbox.value,
            )

        y_win, sr, a0, a1, _ = extract_audio_window(audio_path, t0, t1 - t0)
        if y_win is not None:
            with audio_box:
                print(f"Àudio reproduït: {a0:.2f}s → {a1:.2f}s")
                display(Audio(data=y_win.T if y_win.ndim == 2 else y_win, rate=sr, autoplay=autoplay_checkbox.value))
                display(Javascript("""
                    setTimeout(() => {
                        const audios = document.querySelectorAll("audio");
                        if (audios.length > 0) {
                            const a = audios[audios.length - 1];
                            a.playbackRate = 0.5;
                        }
                    }, 100);
                """))

    for w in [
        start_slider, window_slider, max_vel_slider, dev_slider, expand_slider,
        nav_dropdown, autoplay_checkbox, overlay_counts_checkbox,
        old_vel_checkbox, old_dev_checkbox, new_vel_checkbox, new_shape_checkbox
    ]:
        w.observe(_refresh, names="value")

    controls = widgets.VBox([
        window_slider,
        start_slider,
        widgets.HBox([prev_button, next_button]),
        widgets.HBox([max_vel_slider, dev_slider, expand_slider]),
        widgets.HBox([nav_dropdown, prev_run_button, next_run_button]),
        widgets.HBox([autoplay_checkbox, overlay_counts_checkbox, old_vel_checkbox, old_dev_checkbox, new_vel_checkbox, new_shape_checkbox]),
    ])

    display(controls, info_box, plot_box, audio_box)
    _sync_start_bounds()
    _refresh()
